<a href="https://colab.research.google.com/github/LCaravaggio/NLP/blob/main/notebooks/09_Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vamos a usar [Ollama](https://ollama.com/) con un modelo de lenguaje local.

* Es un programa que permite descargar y ejecutar modelos de lenguaje de manera local.
* Desde la terminal, `ollama pull llama3.1` descarga un modelo y `ollama run llama3.1` abre un chat interactivo.
* También expone una API local en `localhost:11434` que se puede consumir desde cualquier lenguaje. En Python, el paquete `ollama` provee una interfaz directa con funciones como `ollama.chat()` y `ollama.generate()`.

Ollama se instala y ejecuta directamente en la instancia de Colab.
Recomendamos usar un runtime con GPU.

-----------------------

Tarea: responder donde dice **PREGUNTA**

## Configuración del entorno

In [ ]:
!sudo apt-get install -y zstd pciutils # dependencias
!curl -fsSL https://ollama.com/install.sh | sh # Ollama

In [ ]:
!pip install -qU ollama datasets watermark

In [ ]:
%reload_ext watermark

In [ ]:
%watermark -vmp ollama,datasets,numpy,pandas

Hay que iniciar el servidor Ollama en segundo plano y dejarlo corriendo

In [ ]:
import subprocess
import time

# matar cualquier proceso anterior y reiniciar el servidor
!pkill -f "ollama serve" 2>/dev/null

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

print(f"Ollama server iniciado (PID: {process.pid})")

In [ ]:
# Modelo acorde a los recursos de Colab
MODEL_NAME = "gemma3:1b"

!ollama pull {MODEL_NAME}

In [ ]:
import ollama

models = ollama.list()
for m in models['models']:
    print(f"{m['model']}  ({m['size'] / 1e9:.1f} GB)")

En [ollama.com/library](https://ollama.com/library) están todos los modelos con sus variantes.

Para comparar rendimiento entre modelos:

* [Chatbot Arena Leaderboard](https://lmarena.ai/) ranquea modelos según preferencia humana
* [Artificial Analysis](https://artificialanalysis.ai/) publica índices de inteligencia que agregan benchmarks independientes, incluyendo modelos chicos y abiertos

## Introducción al uso de Ollama

Usamos un LLM local para generar texto a partir de prompts.

In [ ]:
%%time
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Qué es una chocotorta?"}]
)

In [ ]:
print(response)

In [ ]:
print(response['message']['content'])

**PREGUNTA 1**: ¿Qué caracteriza a la respuesta anterior? ¿Por qué?

* Ollama carga el modelo en memoria (RAM o VRAM) de manera automática con el primer request.
* El modelo queda cargado durante 5 minutos por defecto; si no recibe requests en ese tiempo, se libera de la memoria automáticamente.
* Para ver qué modelos están cargados y si corren en GPU o CPU (columna `PROCESSOR`), se puede usar `ollama ps`.
* Para liberar un modelo de la memoria manualmente, `ollama stop <modelo>`.
* Si se hace un request con un modelo distinto, Ollama lo carga; si no entra en memoria junto con el anterior, primero libera el anterior.

A veces es útil usar "system prompts" para alterar el comportamiento del modelo.

In [ ]:
system_instruction = """\
Sos un experto riguroso, breve y conciso. Respondés siempre con lo justo y necesario, sin listas, títulos ni negritas.
"""
print(system_instruction)

In [ ]:
%%time
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": "¿Qué es una chocotorta?"}
    ]
)

In [ ]:
print(response['message']['content'])

In [ ]:
# función para hacer un print en formato markdown.
import textwrap

from IPython.display import Markdown

def to_markdown(text):
    text = text.replace('•', '  *').replace("\n", "\n\n")
    # predicate: hack para indentar todas las líneas
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Por qué Sibarita es tan rica?"}]
)
to_markdown(response['message']['content'])

In [ ]:
# Ollama devuelve métricas en cada respuesta.

ns_to_s = 1e-9 # duraciones en nanosegundos

prompt_tokens = response['prompt_eval_count']
generated_tokens = response['eval_count']
prompt_time = response['prompt_eval_duration'] * ns_to_s
generation_time = response['eval_duration'] * ns_to_s
total_time = response['total_duration'] * ns_to_s

print(f"Tokens del prompt:    {prompt_tokens}")
print(f"Tokens generados:     {generated_tokens}")
print(f"Tiempo de prompt:     {prompt_time:.2f}s ({prompt_tokens / prompt_time:.1f} tokens/s)")
print(f"Tiempo de generación: {generation_time:.2f}s ({generated_tokens / generation_time:.1f} tokens/s)")
print(f"Tiempo total:         {total_time:.2f}s")

**PREGUNTA 2** : ¿Qué diferencia hay entre "tiempo de prompt" y "tiempo de generación"? ¿A que se debe la diferencia?

Con el parámetro `options` podemos ajustar parámetros de generación, como `temperature` o `num_predict` (número máximo de tokens en el output).

In [ ]:
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Escribir una canción de cancha para la selección de Uruguay."}],
    options={
        "num_predict": 120,
        "temperature": 1.0,
        "seed": 33, # resultados determinísticos
    }
)

In [ ]:
to_markdown(response['message']['content'])

**PREGUNTA 3**: ¿Cómo podemos obtener resultados determinísticos si temperature=1.0?

In [ ]:
# Con ollama show <modelo> vemos las características de un modelo descargado.
!ollama show {MODEL_NAME}

In [ ]:
# También desde python
info = ollama.show(MODEL_NAME)

for key, value in info.modelinfo.items():
    print(f"{key}: {value}")

## Clasificación zero-shot

Usemos el LLM para clasificar reviews.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('cornell-movie-review-data/rotten_tomatoes')

In [ ]:
print(dataset)

In [ ]:
id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

In [ ]:
examples = dataset['test'].shuffle(seed=36).select(range(5))
examples[:]

In [ ]:
from tqdm import tqdm

def zero_shot_clf(review):
    prompt = f'Classify the following movie review as "positive" or "negative": "{review}\"'
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

responses = []
for review in tqdm(examples['text']):
    response = zero_shot_clf(review)
    responses.append(response)

In [ ]:
for example, response in zip(examples, responses):
    print(f"<<Review>> {example['text']}")
    print(f"<<Response>> {response['message']['content']}")
    print(f"<<Ground truth>> {id2label[example['label']]}")
    print("-"*80)

### Chain-of-thought

Usemos _chain-of-thought_ y un prompt que nos permita encontrar la respuesta final fácilmente.

In [ ]:
import re

def cot_clf(review):
    prompt = (
        '''Classify the following movie review as "positive" or "negative".'''
        ''' First, reason step by step about the review.'''
        ''' Then, write your answer on the last line in the format "Final Classification: [positive/negative]".\n\n'''
        f'''Review: "{review}"'''
    )
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

def extract_final_classification(response_text):
    lines = response_text.split('\n')
    for line in lines:
        if 'Final Classification:' in line:
            answer = line.split(':')[1].strip()
            answer = re.sub(r'[^\w\s]', '', answer.lower())
            return answer
    return None

In [ ]:
responses = []
for review in tqdm(examples['text']):
    response = cot_clf(review)
    responses.append(response)

In [ ]:
def print_results(examples, responses):
    for example, response in zip(examples, responses):
        print(f"Review: {example['text']}")
        text = response['message']['content']
        print(f"<<Response>> {text}")
        print(f"<<Final Classification>> {extract_final_classification(text)}")
        print(f"<<Ground truth>> {id2label[example['label']]}")
        print("-"*80)

print_results(examples, responses)

## Prompt prefilling y stop sequences

Para obtener respuestas acotadas y parseables, podemos usar:

* **Prompt prefilling** consiste en pre-llenar el inicio de la respuesta del asistente para guiar la generación. En `chat()`, se agrega un mensaje con `"role": "assistant"` al final de la lista.
* Las **stop sequences** son strings que, al aparecer en la generación, hacen que el modelo deje de generar.

El prefilling es especialmente útil para forzar un formato de salida (JSON, XML, código) sin depender de que el modelo siga las instrucciones del prompt. La stop sequence complementa esto evitando que el modelo siga generando después del cierre esperado.

A continuación comparamos una respuesta sin estas técnicas con otra que combina ambas.

In [ ]:
# Sin prefilling ni stop: el modelo responde libremente
review = "The film was so nice that I almost vomited."
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": f'Classify this movie review as "positive" or "negative".\n\nReview: "{review}"'}
    ]
)
print(response['message']['content'])

In [ ]:
# Con prefilling + stop sequence:
# - Prefilling: forzamos que el modelo arranque razonando
# - Stop: cortamos en ``` para obtener solo las dos líneas
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": (
            f'Classify this movie review as "positive" or "negative".\n\n'
            f'Respond in exactly this format:\n'
            f'```\nReasoning: <brief reasoning>\n'
            f'Classification: <positive or negative>\n```\n\n'
            f'Review: "{review}"'
        )},
        {"role": "assistant", "content": "Reasoning: "}
    ],
    options={"stop": ["```"]}
)

print(response["message"]["content"])

## Clasificación few-shot

Puede ser útil pasar ejemplos de cómo se resuelve la tarea. En este caso elegimos los ejemplos aleatoriamente del set de entrenamiento y escribimos a mano el razonamiento de cada uno. También podemos elegir los ejemplos manualmente, o buscando ejemplos parecidos en el set de entrenamiento.

In [ ]:
from datasets import concatenate_datasets

# Razonamientos escritos a mano para los ejemplos que selecciona get_few_shot_examples
reasonings = {
    "[a] shapeless blob of desperate entertainment .":
        "The reviewer calls the film a \"shapeless blob\" and describes the entertainment as \"desperate\", which are dismissive judgments with nothing positive to balance them.",
    "films about loss , grief and recovery are pretty valuable these days . seen in that light , moonlight mile should strike a nerve in many .":
        "The reviewer considers films about loss and grief valuable and expects Moonlight Mile to resonate with many viewers, which is an endorsement.",
    "it's super- violent , super-serious and super-stupid .":
        "The list of exaggerated descriptors ends with \"super-stupid\", which makes it clear that the review is mocking the film.",
    "ruzowitzky has taken this mothball-y stuff and made a rather sturdy , old-fashioned entertainment out of it .":
        "Although the source material is described as dated (\"mothball-y\"), the reviewer praises the director for turning it into sturdy, old-fashioned entertainment.",
}

def get_few_shot_examples(dataset, num_examples=4):
    examples_pos = dataset['train'].filter(lambda x: x['label'] == 1).shuffle(seed=36).select(range(num_examples // 2))
    examples_neg = dataset['train'].filter(lambda x: x['label'] == 0).shuffle(seed=36).select(range(num_examples // 2))
    examples = concatenate_datasets([examples_pos, examples_neg]).shuffle(seed=5)
    out = []
    for example in examples:
        label = id2label[example['label']]
        formatted_example = (
            f'''Review: "{example['text']}"'''
            f'''\nReasoning: {reasonings[example['text']]}'''
            f'''\nFinal Classification: {label}'''
        )
        out.append(formatted_example)
    return out

few_shot_examples = get_few_shot_examples(dataset)
for example in few_shot_examples:
    print(example)
    print("-"*80)

**PREGUNTA 4**: ¿De dónde se toman los ejemplos en few-shot? ¿Train, validation, test?

In [ ]:
def cot_few_shot_prompt(review, few_shot_examples):
    prompt = (
        '''Classify the following movie review as "positive" or "negative".'''
        ''' First, reason step by step about the review.'''
        ''' Then, write your answer on the last line in the format "Final Classification: [positive/negative]".\n\n'''
    )
    prompt += 'Examples:'
    prompt += '\n------\n'
    prompt += '\n\n'.join(few_shot_examples)
    prompt += '\n------\n'
    prompt += f'\nReview: "{review}"'
    return prompt

print(cot_few_shot_prompt("EJEMPLO.", few_shot_examples))

In [ ]:
def cot_few_shot_clf(review, few_shot_examples):
    prompt = cot_few_shot_prompt(review, few_shot_examples)
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

In [ ]:
responses = []
for review in tqdm(examples['text']):
    response = cot_few_shot_clf(review, few_shot_examples)
    responses.append(response)

In [ ]:
print_results(examples, responses)

**PREGUNTA 5** ¿Cómo probarías en un caso real qué tipo de prompt o qué modelo es mejor?

## Extracción de datos estructurados de texto libre

Veamos cómo extraer los ingredientes de una receta.

In [ ]:
ds_recetas = load_dataset("somosnlp/RecetasDeLaAbuela", "version_1")

In [ ]:
print(ds_recetas)

In [ ]:
ds_small = ds_recetas["train"].filter(lambda x: x["Pais"] == "ARG").shuffle(seed=33).select(range(5))

In [ ]:
import ast
import re

def preprocess(example):
    if example["Pasos"].startswith("["):
        pasos_list = ast.literal_eval(example["Pasos"].encode("unicode_escape").decode("ascii"))
        example["Pasos"] = " ".join(pasos_list)
    example["Pasos"] = re.sub(r'\s+', ' ', example["Pasos"])
    return example

ds_small = ds_small.map(preprocess)

In [ ]:
for example in ds_small:
    print(example["Ingredientes"])
    print(example["Pasos"])
    print("-"*80)

In [ ]:
def prompt_extract_json(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

print(prompt_extract_json(ds_small[0]["Pasos"]))

In [ ]:
import json

# Los modelos chicos a veces se traban repitiendo el mismo token con format="json"
# (por ejemplo, espacios o saltos de línea). Ollama corta la generación y lanza un
# ResponseError ("token repeat limit reached"). Capturamos el error y devolvemos una
# respuesta vacía con la misma forma, para que el resto del procesamiento siga funcionando.
def safe_chat(**kwargs):
    try:
        return ollama.chat(**kwargs)
    except ollama.ResponseError as e:
        print(f"Error de Ollama: {e.error}")
        return {"message": {"content": ""}}

def extract_json(receta):
    prompt = prompt_extract_json(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}  # tope de tokens por si el modelo se traba
    )
    return response

In [ ]:
responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json(receta)
    responses.append(response)

In [ ]:
for example, response in zip(ds_small, responses):
    print(f"Receta: {example['Pasos']}")
    print(f"Response: {response['message']['content']}")
    print("-"*80)

In [ ]:
def prompt_extract_json2(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener únicamente la clave "ingredientes".'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json2(receta):
    prompt = prompt_extract_json2(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}
        # constrained decoding: solo permite tokens que mantengan el texto como un JSON válido.
        # Si la generación se corta antes de cerrar el JSON, el resultado puede quedar incompleto.
        # Para fijar también las claves y los tipos se puede pasar un JSON schema:
        # format=Modelo.model_json_schema() (con una clase de pydantic)
        # La alternativa menos segura es usar prefilling y stop sequences
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json2(receta)
    responses.append(response)

In [ ]:
results = []
for response in responses:
    try:
        data = json.loads(response['message']['content'])
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
results[0]

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.head()

In [ ]:
df['ingredientes'].explode().value_counts()

In [ ]:
def prompt_extract_json3(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener las claves "veganos" y "no_veganos".'''
        ''' Ambas claves deben contener una lista de ingredientes.'''
        ''' Si no hay ingredientes para alguna clave, se debe incluir una lista vacía.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json3(receta):
    prompt = prompt_extract_json3(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json3(receta)
    responses.append(response)

In [ ]:
results = []
for response in responses:
    try:
        data = json.loads(response['message']['content'])
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

df = pd.DataFrame(results)
df.head()

In [ ]:
df['veganos'].explode().value_counts()
# Recomendación: en general es mejor descomponer las tareas en pasos
# Acá: primero identificar componentes, luego clasificar en vegano/no-vegano

## Conversaciones multi-turno

Con Ollama manejamos el historial de mensajes manualmente.

In [ ]:
history = []

def chat_send(message, history):
    history.append({"role": "user", "content": message})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=history
    )
    assistant_msg = response['message']['content']
    history.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg

In [ ]:
print(f"{len(history)=}")
response_text = chat_send("¿Por qué el agua es azul?", history)
to_markdown(response_text)

In [ ]:
print(f"{len(history)=}")
response_text = chat_send("¿Cómo se lo explicarías a un niño de 8 años?", history)
to_markdown(response_text)

In [ ]:
print(f"{len(history)=}")
for message in history:
    display(to_markdown(f"**{message['role']}**: {message['content']}"))

**PREGUNTA 6**: ¿Qué pasa si en chat_send no agregamos la respuesta del asistente al historial?

## `generate()` vs. `chat()`

Hay dos interfaces para hacer inferencia con los modelos:

* `ollama.generate()` recibe un string como prompt. Por defecto, Ollama le aplica el template de chat del modelo (como si fuera un único mensaje del usuario); con `raw=True` el string se pasa tal cual y el modelo genera una continuación (completion).
* `ollama.chat()` es la interfaz conversacional: recibe una lista de mensajes con roles (`system`, `user`, `assistant`) y el modelo responde respetando el template de chat con el que fue entrenado.
* `chat()` es más cómodo para conversaciones multi-turno
* `generate()` con raw=True es útil cuando se quiere control total sobre el prompt exacto que recibe el modelo. Cada familia de modelos usa un formato distinto.
* Para hacer prompt prefilling en `generate()`, se usa `raw=True` y se incluye el inicio de la respuesta al final del prompt, después de la marca de turno del modelo.

In [ ]:
# El template del modelo:
!ollama show --modelfile {MODEL_NAME} | grep -A 20 "TEMPLATE"

In [ ]:
# Define cómo Ollama convierte la lista de mensajes en el prompt que recibe el modelo.
# * Si el rol es user o system → lo envuelve en <start_of_turn>user ... <end_of_turn>
# * Si el rol es assistant → lo envuelve en <start_of_turn>model ... <end_of_turn>
# * Si el último msje es del usuario, agrega <start_of_turn>model al final
#   (para que el modelo empiece a generar su turno)
# * Si el último es del assistant, no cierra con <end_of_turn>
#   (para que el modelo siga desde ahí, esto es lo que habilita el prefilling)
# * El PARAMETER stop le dice a Ollama que corte la generación cuando el modelo
#   genera <end_of_turn>, para que no invente turnos adicionales.

In [ ]:
# chat(): recibe una lista de mensajes con roles
response_chat = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Qué es la física cuántica?"}]
)
print(response_chat['message']['content'])



In [ ]:
# generate() sin raw=True: Ollama aplica el template de chat al prompt,
# por eso la salida es una respuesta y no una continuación del texto
response_gen = ollama.generate(
    model=MODEL_NAME,
    prompt="¿Qué es la física cuántica?"
)
print(response_gen['response'])

In [ ]:
# Para Gemma, el template es:
# <start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n
prompt_con_marcas = """<start_of_turn>user
¿Qué es la física cuántica?<end_of_turn>
<start_of_turn>model
"""

response = ollama.generate(
    model=MODEL_NAME,
    prompt=prompt_con_marcas,
    raw=True  # raw=True evita que Ollama aplique el template de nuevo
)
print(response['response'])

----------------------------

## Extras

* **Streaming**: con `stream=True` podemos recibir la respuesta token a token en lugar de esperar a que se genere completa. Esto es útil para interfaces interactivas.

* **Cuantización**: los modelos en la library de Ollama suelen estar disponibles en distintos niveles de cuantización: `q4_K_M`, `q8_0`, `fp16`, etc. Esto reduce la precisión numérica de los pesos del modelo (por ejemplo, de 16 bits a 4 bits), lo que disminuye el uso de memoria y acelera la inferencia a costa de una leve pérdida de calidad. En la práctica, las variantes `q4` suelen ser un buen compromiso: ocupan ~3x menos memoria que `fp16` con una degradación muy leve en la mayoría de las tareas.

Por ejemplo:

```bash
ollama pull gemma3:1b              # cuantización default (q4_K_M)
ollama pull gemma3:1b-it-fp16      # precisión completa
ollama pull gemma3:1b-it-q8_0      # cuantización a 8 bits
```

* Ollama también tiene modelos **multimodales** que pueden recibir imágenes además de texto. Se pasan las imágenes en el parámetro `images` de `ollama.chat()`.

* También se pueden generar embeddings con `ollama.embed()`. En principio podemos obtener los embeddings de cualquier modelo, pero esto tiene sentido cuando queremos vectores optimizados para una tarea basada en similitud semántica; e.g. clustering.

* **[vLLM](https://docs.vllm.ai/)** es un motor de inferencia optimizado para alto throughput. Ollama está pensado para uso local e interactivo (puede atender algunos requests en paralelo, configurable con `OLLAMA_NUM_PARALLEL`), mientras que vLLM está pensado para servir modelos en producción o procesar grandes volúmenes de datos. Su ventaja principal es el **batching eficiente**: con PagedAttention la KV cache se asigna en bloques según los tokens que realmente usa cada secuencia, y con continuous batching puede procesar cientos de prompts en paralelo aprovechando la GPU al máximo. Para tareas como clasificar 1000 reviews, vLLM es significativamente más rápido que hacer 1000 llamadas secuenciales a Ollama. El parámetro `max_model_len` fija la longitud máxima (prompt + respuesta) de cada secuencia; vLLM verifica al iniciar que una secuencia de esa longitud entre en la KV cache, así que conviene ajustarlo a lo que necesita la tarea en lugar de usar el default del modelo. En nuestro caso, las reviews tienen ~25 palabras en promedio, así que `max_model_len=1024` es más que suficiente.